<a href="https://colab.research.google.com/github/Selvapriya05/Selvapriya-Codeboosters-2026/blob/main/Day_8/Day_8_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers chromadb groq pandas -q



In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

from groq import Groq
import os

print("All libraries imported successfully")

All libraries imported successfully


In [ ]:
GROQ_API_KEY = "GROQ_API_KEY"

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized.")
print("Note: If you see an authentication error later,double check your API key.")

Groq API client initialized.
Note: If you see an authentication error later,double check your API key.


In [ ]:
df = pd.read_csv('college_notes.csv')
print("Shape of dataset:", df.shape)
print("\nColumn names:", df.columns.tolist())

print("\nFirst 3 rows:")
print(df.head(3))

Shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']

First 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [ ]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())

print("\nSample of topics:")

print(df[['note_id','subject', 'topic']].to_string(index=False))

print("\nLength of content (number of characters) for each note:")

df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))

Subjects in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python

In [ ]:
documents = df['content'].tolist()

ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
     for row in df.to_dict('records')
]

print(f"Total chunks prepared : {len(documents)}")
print(f"First document ID     : {ids[0]}")
print(f"First metadata        : {metadatas[0]}")
print(f"First 100 chars of doc: {documents[0][:100]}...")

Total chunks prepared : 15
First document ID     : note_N001
First metadata        : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [ ]:
print("Loading embedding model...")
print("(This may take 30-60 seconds on first run-model is being downloaded)")
print("(Subsequent runs will be faster as the model is cached)")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded successfully.")

test_embedding = embedding_model.encode("This is a sentence")
print(f"Test embedding shape: {test_embedding.shape}")

print(f"First 5 values of test embedding: {test_embedding[:5]}")

Loading embedding model...
(This may take 30-60 seconds on first run-model is being downloaded)
(Subsequent runs will be faster as the model is cached)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully.
Test embedding shape: (384,)
First 5 values of test embedding: [ 0.05048309  0.088006    0.00487488  0.03626884 -0.00101813]


In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="college_notes_rag")

print("ChromaDB client created.")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


ChromaDB client created.
Collection name: college_notes_rag
Documents in collection so far: 15


In [ ]:
print("Generating embeddings for all 15 notes...")
print("This may take 15-30 seconds...")

embeddings = embedding_model.encode(documents, show_progress_bar=True)

print(f"\nEmbedding matrix shape: {embeddings.shape}")

embeddings_list = embeddings.tolist()

collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas
)

print(f"\nDocuments successfully added to ChromaDB.")
print(f"Total document in collection: {collection.count()}")

Generating embeddings for all 15 notes...
This may take 15-30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given



Embedding matrix shape: (15, 384)

Documents successfully added to ChromaDB.
Total document in collection: 15


In [ ]:
def retrieve_relevant_chunks(question, top_k=3):
    """
    Given a user question, retrieve the most relevant document
    """

    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    return results


print("Retrieval function defined successfully.")
print("Function: retrieve_relevant_chunks(question, top_k=3)")

Retrieval function defined successfully.
Function: retrieve_relevant_chunks(question, top_k=3)


In [ ]:
test_question = "What is ETL and how does it work in data engineering?"
print(f"Test Question: {test_question}")
print("=" * 60)


results = retrieve_relevant_chunks(test_question,top_k=3)

print("\nTop 3 Retrieved chunks:")
print("=" * 60)

for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):

   print(f"\nResult {i+1}:")
   print(f"  Subject  : {meta['subject']}")
   print(f"  Topic    : {meta['topic']}")
   print(f"  Distance : {dist:.4f}")
   print(f"  Content  : {doc[:120]}...")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Test Question: What is ETL and how does it work in data engineering?

Top 3 Retrieved chunks:

Result 1:
  Subject  : Data Engineering
  Topic    : ETL Pipelines
  Distance : 0.2269
  Content  : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Result 2:
  Subject  : Data Engineering
  Topic    : APIs and Data Collection
  Distance : 1.0690
  Content  : An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result 3:
  Subject  : Python Programming
  Topic    : Data Visualization
  Distance : 1.3375
  Content  : Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


In [ ]:
def build_context_from_results(results):
  """
  Format ChromaDB retrieval results into a readable context string

  Parameters:
      results: The output from collection.query() - a dictionary

  Returns:
      context_str (str) : A formatted string of all retrieved document chunks

  """
  context_parts = []

  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
      chunk_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}]\n{doc}"


In [ ]:
# def generate_rag_answer(question,context):
#    """
#    Send the retrieved context and question to the Groq LLM for answer generation.

#    Parameters:
#         question (str) : The user's question
#         context (str)  : The retrieved context chunks (formatted string)

#    Returns:
#         answer (str) : The LLM-generated answer to the question
#    """

#    system_prompt = """You are a helpful academic assistant for engineering students.

#    You will be given context retrieved from a college knowledge base, and a

#    RULES:
#    1.Answer ONLY using the information provided in the context below.
#    2.If the answer is not found in the context, say exactly:
#      "I don't
#    """

#    user_prompt = f"""Context from knowledge

#   """
#   response = groq_client.chat.completions.create(
#       model="llama-3.1-8b-instant",
#       messages=[
#           {"role": "system", "content": system_prompt},
#           {"role": "user", "content": user_prompt}
#       ],
#       temperature = 0.1
#       max_tokens=500
#   )

#   answer = response.choices[0].message.content
#   return answer

#   print("RAG generation function defined successfully.")


